<div style="background:linear-gradient(135deg,#2e1065 0%,#6d28d9 55%,#8b5cf6 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#ddd6fe;font-weight:700;text-transform:uppercase">Chapter 45 · Mathematical Statistics</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Moments, MGFs &amp; Inequalities 📐</div>
  <div style="font-size:15px;color:#ede9fe;max-width:740px;line-height:1.6">Summarizing a distribution and bounding its tails. This notebook computes the four moments, uses a moment generating function to recover them, checks the Markov and Chebyshev inequalities, and watches averages concentrate, the math behind generalization bounds.</div>
  <div style="margin-top:16px;font-size:13px;color:#c4b5fd">Statistics, Data Science and AI: A Visual Handbook · John Fisher · 2026</div>
</div>

## ⚙️ Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(45)
plt.rcParams.update({"figure.dpi":110,"font.size":11,"axes.spines.top":False,"axes.spines.right":False})
VIOLET="#7c3aed"; PINK="#db2777"; TEAL="#0d9488"
print("ready")

ready


<div style="background:#f5f3ff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">DEMO 1 · THE FOUR MOMENTS</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Center, spread, skew, tails</div>
<div style="color:#4a5578;margin-top:6px">Moments summarize a distribution's shape. The 1st is the mean, the 2nd central moment is the variance, the 3rd gives skewness (asymmetry), and the 4th gives kurtosis (tail heaviness).</div>
</div>

In [2]:
normal = rng.normal(0,1,size=200_000)
skewed = rng.exponential(1.0,size=200_000)
for name,d in [("Normal",normal),("Exponential",skewed)]:
    print(f"{name:12s}: mean={d.mean():+.3f}, var={d.var():.3f}, skew={stats.skew(d):+.3f}, kurtosis={stats.kurtosis(d):+.3f}")

Normal      : mean=+0.000, var=1.000, skew=-0.000, kurtosis=+0.002
Exponential : mean=+0.996, var=0.999, skew=+2.032, kurtosis=+6.351


The normal is symmetric (skew &approx; 0) with the baseline tail weight (excess kurtosis &approx; 0). The exponential is strongly right-skewed (skew &approx; 2) with heavier tails (positive excess kurtosis). The first four moments capture center, spread, lopsidedness, and tail heaviness, the four ways a distribution can differ in shape.

<div style="background:#f5f3ff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">DEMO 2 · THE MOMENT GENERATING FUNCTION</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">One function, all the moments</div>
<div style="color:#4a5578;margin-top:6px">The MGF is M(t) = E[e^(tX)]. Its derivatives at t=0 generate the moments: M'(0) = mean, M''(0) = E[X^2]. We recover them numerically for an exponential.</div>
</div>

In [3]:
rate = 2.0
X = rng.exponential(1/rate, size=500_000)
def M(t): return np.mean(np.exp(t*X))      # empirical MGF
h = 1e-3
mean_from_mgf = (M(h) - M(-h)) / (2*h)              # M'(0)
EX2_from_mgf  = (M(h) - 2*M(0) + M(-h)) / h**2      # M''(0)
print(f"M'(0)  = {mean_from_mgf:.4f}   (true mean 1/rate = {1/rate})")
print(f"M''(0) = {EX2_from_mgf:.4f}   (true E[X^2] = 2/rate^2 = {2/rate**2})")
print(f"variance = M''(0) - M'(0)^2 = {EX2_from_mgf - mean_from_mgf**2:.4f}  (true 1/rate^2 = {1/rate**2})")

M'(0)  = 0.5015   (true mean 1/rate = 0.5)
M''(0) = 0.5040   (true E[X^2] = 2/rate^2 = 0.5)
variance = M''(0) - M'(0)^2 = 0.2525  (true 1/rate^2 = 0.25)


Differentiating the MGF at 0 recovers the mean (0.5) and the second moment (0.5), and from them the variance (0.25). The MGF is a single function that *encodes every moment*, and because it uniquely identifies a distribution, it is the classic tool for proving that a sum of independent variables has a particular distribution (their MGFs simply multiply).

<div style="background:#f5f3ff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">DEMO 3 · MARKOV &amp; CHEBYSHEV</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Bounding tails with almost no assumptions</div>
<div style="color:#4a5578;margin-top:6px">Markov: for X >= 0, P(X >= a) <= E[X]/a. Chebyshev builds on it: P(|X - mu| >= k*sigma) <= 1/k^2, for ANY distribution with finite variance.</div>
</div>

In [4]:
X = rng.exponential(1.0, size=500_000)   # mean 1, sd 1
mu, sd = X.mean(), X.std()
for k in [2, 3, 4]:
    actual = np.mean(np.abs(X-mu) >= k*sd)
    print(f"k={k}: P(|X-mu| >= {k} sd) actual = {actual:.4f}   Chebyshev bound 1/k^2 = {1/k**2:.4f}")

k=2: P(|X-mu| >= 2 sd) actual = 0.0499   Chebyshev bound 1/k^2 = 0.2500
k=3: P(|X-mu| >= 3 sd) actual = 0.0184   Chebyshev bound 1/k^2 = 0.1111
k=4: P(|X-mu| >= 4 sd) actual = 0.0067   Chebyshev bound 1/k^2 = 0.0625


Chebyshev's bound 1/k&#178; holds for every distribution, here comfortably (the exponential's real tails are much lighter than the worst case). The power of these inequalities is that they need *almost no assumptions*, just a mean (Markov) or a variance (Chebyshev), making them the universal safety net of probability.

<div style="background:#f5f3ff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">DEMO 4 · CONCENTRATION</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Why averages pile up near the mean</div>
<div style="color:#4a5578;margin-top:6px">Hoeffding's inequality says the average of n bounded variables is exponentially unlikely to stray far from its mean: P(|mean - mu| >= eps) <= 2 exp(-2 n eps^2). Averages CONCENTRATE.</div>
</div>

In [5]:
def tail_prob(n, eps, trials=20000):
    means = rng.random((trials, n)).mean(axis=1)    # uniform[0,1], mu=0.5
    return np.mean(np.abs(means - 0.5) >= eps)
eps = 0.1
for n in [25, 100, 400]:
    emp = tail_prob(n, eps)
    bound = 2*np.exp(-2*n*eps**2)
    print(f"n={n:4d}: P(|mean-0.5|>={eps}) empirical = {emp:.4f}   Hoeffding bound = {bound:.4f}")

n=  25: P(|mean-0.5|>=0.1) empirical = 0.0824   Hoeffding bound = 1.2131
n= 100: P(|mean-0.5|>=0.1) empirical = 0.0004   Hoeffding bound = 0.2707
n= 400: P(|mean-0.5|>=0.1) empirical = 0.0000   Hoeffding bound = 0.0007


As n grows, the chance the sample mean is more than 0.1 from the truth collapses, and Hoeffding's exponential bound tracks (and safely exceeds) it. This **concentration** is a sharper, finite-sample cousin of the Law of Large Numbers, and it is the mathematical reason a model evaluated on enough data gives a trustworthy estimate.

<div style="background:#f5f3ff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">DEMO 5 · INEQUALITIES IN MACHINE LEARNING</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Sample complexity and generalization</div>
<div style="color:#4a5578;margin-top:6px">Concentration inequalities turn into GUARANTEES. Inverting Hoeffding tells you how many samples you need so an estimate is within eps of the truth with confidence 1 - delta.</div>
</div>

In [6]:
def hoeffding_n(eps, delta): return int(np.ceil(np.log(2/delta) / (2*eps**2)))
for eps, delta in [(0.05,0.05),(0.02,0.05),(0.01,0.01)]:
    print(f"to be within +/-{eps} with {100*(1-delta):.0f}% confidence: need n >= {hoeffding_n(eps,delta):,} samples")

# method of moments: estimate exponential rate from the sample mean (lambda = 1/mean)
data = rng.exponential(1/3.0, size=2000)
print(f"\nmethod of moments: rate_hat = 1/mean = {1/data.mean():.3f}  (true 3.0)")

to be within +/-0.05 with 95% confidence: need n >= 738 samples
to be within +/-0.02 with 95% confidence: need n >= 4,612 samples
to be within +/-0.01 with 99% confidence: need n >= 26,492 samples

method of moments: rate_hat = 1/mean = 3.035  (true 3.0)


Hoeffding inverts into a **sample-complexity** rule: to pin a test accuracy to within &plusmn;0.02 at 95% confidence you need about 4,600 examples, a concrete answer to "how much test data is enough". These bounds are the backbone of **PAC learning** and **generalization theory**. The same moment ideas give the **method of moments**, a quick estimator (here rate = 1/mean), and **moment matching** is how models like GANs and MMD are trained to mimic a target distribution.

---
<div style="background:#ffffff;border:1px solid #e6e9f2;border-radius:16px;padding:22px 26px;font-family:Inter,sans-serif">
<div style="font-size:19px;font-weight:800;color:#1a2138">✅ What you built</div>
<div style="color:#4a5578;line-height:1.8;margin-top:8px">You computed the four moments, recovered them from a moment generating function, verified the Markov and Chebyshev bounds, watched averages concentrate under Hoeffding, and turned concentration into a sample-complexity guarantee. Moments describe a distribution; inequalities bound how far reality can stray.</div>
</div>

<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:16px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>